# Activity Day 17: const references, Rule of Three, operator overloading

### Question 2

Change the `printRoster` function so that it (1) avoids copying the entire vector and (2) guarantees it won't modify the roster.

**(a)** Write the new function signature.

**(b)** Does the code still compile after your change? If not, what else needs to change and why?

In [1]:
////q2.cpp
//% CXXFLAGS -std=c++11
#include<iostream>
#include<vector>
#include<string>

class Student {
private:
    std::string name_;
    double gpa_;
public:
    Student(std::string n, double g){
        this->name_ = n; this->gpa_ = g;
    }
    std::string getName() const {
        return this->name_;
    }
    double getGpa() const {
        return this->gpa_;
    }
};

void printRoster(const std::vector<Student>& roster) {
    for (int i = 0; i < roster.size(); i++) {
        std::cout << roster[i].getName() << ": "
                  << roster[i].getGpa() << "\n";
    }
}

int main(){
    std::vector<Student> cs4300 = {Student("Alice", 3.8), Student("Bob", 3.5), Student("Carol", 3.9)};
    printRoster(cs4300);
}

wrote file q2.cpp
$> g++  -std=c++11 q2.cpp    -o q2
$> ./q2 
Alice: 3.8
Bob: 3.5
Carol: 3.9


### Question 3

The `Roster` class below needs the Rule of Three. Write the destructor, copy constructor, and copy assignment operator. Then run the test in main to verify your deep copy works.

In [1]:
////q3.cpp
#include<iostream>
#include<string>

class Roster{
private:
    std::string* names_;
    int size_;
public:
    Roster(int n){
        this->size_ = n;
        this->names_ = new std::string[n];
    }
    
    // Write the destructor:
    ~Roster(){
        delete[] this->names_;
    }
    
    // Write the copy constructor:
    Roster(const Roster& other){
        this->size_ = other.size_;
        this->names_ = new std::string [other.size_];
        for(int i = 0; i < other.size_; i++){
            this->names_[i] = other.names_[i];
        }
    }
    
    // Write the copy assignment operator:
    Roster& operator=(const Roster& other){
        if(this != &other){
            delete[] this->names_;
            this -> size_ = other.size_;
            this->names_ = new std::string [other.size_];
            for(int i =0; i < other.size_; i++){
                this->names_[i] = other.names_[i]; 
            }
        }
        return *this;
    }
    
    void setName(int i, std::string name){
        this->names_[i] = name;
    }
    void print() const {
        for(int i = 0; i < this->size_; i++){
            std::cout << this->names_[i] << "\n";
        }
    }
};

int main(){
    Roster r1(3);
    r1.setName(0, "Alice");
    r1.setName(1, "Bob");
    r1.setName(2, "Carol");
    
    Roster r2 = r1;        // copy constructor
    r2.setName(0, "Zara");
    
    std::cout << "r1:\n";
    r1.print();  // Should still be Alice, Bob, Carol
    std::cout << "r2:\n";
    r2.print();  // Should be Zara, Bob, Carol
    
    Roster r3(1);
    r3.setName(0, "Nobody");
    r3 = r1;               // copy assignment
    r3.setName(1, "Eve");
    
    std::cout << "r1 after assignment:\n";
    r1.print();  // Should still be Alice, Bob, Carol
    std::cout << "r3:\n";
    r3.print();  // Should be Alice, Eve, Carol
}

wrote file q3.cpp
$> g++   q3.cpp    -o q3
$> ./q3 
r1:
Alice
Bob
Carol
r2:
Zara
Bob
Carol
r1 after assignment:
Alice
Bob
Carol
r3:
Alice
Eve
Carol


### Question 4

Consider the `Character` class with `int* history_` (a heap-allocated array of size 10). The constructor allocates `history_` with `new int[10]`.

**(a)** Draw a memory diagram showing the stack and the heap. Where does the `Character` object live? Where does its `history_` array live?

**(b)** When we call `delete ptr`, what happens step by step?

Run the code below to see the addresses and verify your diagram.

In [ ]:
////q4.cpp
#include<iostream>
#include<string>

class Character{
public:
    std::string name_;
    int coins_;
    int* history_;
    int history_count_;

    Character(std::string n){
        this->name_ = n; this->coins_ = 0;
        this->history_ = new int[10];
        this->history_count_ = 0;
    }
    void collect_coins(int num_coins){
        this->coins_ += num_coins;
        this->history_[this->history_count_] = num_coins;
        this->history_count_++;
    }
    void display() const {
        std::cout << this->name_ << ": " << this->coins_ << " coins\n";
    }
    ~Character(){
        std::cout << "Destructor called for " << this->name_ << "\n";
        std::cout << "  Freeing history_ at " << this->history_ << "\n";
        delete[] this->history_;
    }
};

int main(){
    Character* ptr = new Character("Mario");
    ptr->collect_coins(10);
    ptr->collect_coins(25);
    
    std::cout << "ptr (on stack) = " << &ptr << "\n";
    std::cout << "Character object (on heap) at " << ptr << "\n";
    std::cout << "history_ array (on heap) at " << ptr->history_ << "\n";
    
    ptr->display();
    delete ptr;  // What happens here? Watch the output!
}

### Question 5(a)

Add an `operator*` method to the `Fraction` class below. Recall: (a/b) × (c/d) = (ac)/(bd).

In [ ]:
////q5a.cpp
#include<iostream>

class Fraction{
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}

    int getNum() const { return this->num_; }
    int getDen() const { return this->den_; }

    Fraction operator+(const Fraction& other) const {
        int newNum = this->num_ * other.den_
                   + other.num_ * this->den_;
        int newDen = this->den_ * other.den_;
        return Fraction(newNum, newDen);
    }

    bool operator==(const Fraction& other) const {
        return this->num_ * other.den_
            == other.num_ * this->den_;
    }
    
    // Add operator* here:
    
    

    friend std::ostream& operator<<(std::ostream& out,
                                     const Fraction& f){
        out << f.num_ << "/" << f.den_;
        return out;
    }
};

int main(){
    Fraction a(2, 3);
    Fraction b(3, 4);
    Fraction c = a * b;
    std::cout << a << " * " << b << " = " << c << "\n";
    // Should print: 2/3 * 3/4 = 6/12
}

### Question 5(b)

**Predict the output** of each line before running the code. Then run it to check!

In [ ]:
////q5b.cpp
#include<iostream>

class Fraction{
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}

    Fraction operator+(const Fraction& other) const {
        int newNum = this->num_ * other.den_ + other.num_ * this->den_;
        int newDen = this->den_ * other.den_;
        return Fraction(newNum, newDen);
    }

    bool operator==(const Fraction& other) const {
        return this->num_ * other.den_ == other.num_ * this->den_;
    }

    friend std::ostream& operator<<(std::ostream& out, const Fraction& f){
        out << f.num_ << "/" << f.den_;
        return out;
    }
};

int main(){
    Fraction x(1, 3);
    Fraction y(1, 6);
    Fraction z = x + y;

    std::cout << z << "\n";                     // Line 1: your prediction?
    std::cout << (z == Fraction(1, 2)) << "\n";  // Line 2: your prediction?
    std::cout << (z == Fraction(2, 3)) << "\n";  // Line 3: your prediction?
}

### Question 6

For each class below, does it need the Rule of Three? You can test by creating objects, copying them, and seeing what happens.

**(a)** `Point` — no heap memory, just two doubles.

In [ ]:
////q6a.cpp
#include<iostream>

class Point {
    double x_, y_;
public:
    Point(double x, double y) : x_(x), y_(y) {}
    void display() const { std::cout << "(" << x_ << ", " << y_ << ")\n"; }
};

int main(){
    Point p1(3.0, 4.0);
    Point p2 = p1;  // copy
    // Both should work fine -- no issues
    p1.display();
    p2.display();
}

**(b)** `Message` — has a destructor but no copy constructor or assignment operator. Try copying it!

In [ ]:
////q6b.cpp
#include<iostream>
#include<cstring>

class Message {
    char* text_;
    int length_;
public:
    Message(const char* t){
        length_ = strlen(t);
        text_ = new char[length_ + 1];
        strcpy(text_, t);
    }
    ~Message(){
        std::cout << "Destroying message: " << text_ << " at " << (void*)text_ << "\n";
        delete[] text_;
    }
    void display() const { std::cout << text_ << "\n"; }
};

int main(){
    Message m1("Hello");
    Message m2 = m1;  // shallow copy! both point to same char array
    
    std::cout << "m1: "; m1.display();
    std::cout << "m2: "; m2.display();
    // Watch what happens when destructors run...
    // (This may crash due to double-free!)
}

**(c)** `Playlist` — uses `std::vector`, which manages its own memory.

In [ ]:
////q6c.cpp
//% CXXFLAGS -std=c++11
#include<iostream>
#include<vector>
#include<string>

class Playlist {
    std::vector<std::string> songs_;
public:
    void addSong(std::string s){ songs_.push_back(s); }
    void display() const {
        for(int i = 0; i < songs_.size(); i++)
            std::cout << songs_[i] << "\n";
    }
};

int main(){
    Playlist p1;
    p1.addSong("Song A");
    p1.addSong("Song B");
    
    Playlist p2 = p1;  // copy
    p2.addSong("Song C");
    
    std::cout << "p1:\n"; p1.display();  // Should NOT have Song C
    std::cout << "p2:\n"; p2.display();  // Should have Song C
}

### Question 7 — Challenge: Complex Number Class

Create operator overloads for the `Compl` class. The starter class is provided below.

**(a)** `operator+`: $(a+bi) + (c+di) = (a+c) + (b+d)i$

**(b)** `operator*`: $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$

**(c)** `operator==`: equal if both real and imaginary parts match

**(d)** Bonus: `operator<<` to print as `a + bi` or `a - bi` (requires `friend`)

In [ ]:
////q7.cpp
#include<iostream>

class Compl {
private:
    double re_;
    double im_;
public:
    Compl(double r = 0, double i = 0) : re_(r), im_(i) {}

    double getRe() const { return this->re_; }
    double getIm() const { return this->im_; }

    // (a) Write operator+ here:
    
    
    // (b) Write operator* here:
    
    
    // (c) Write operator== here:
    
    
    // (d) Bonus: Write friend operator<< here:
    
};

int main(){
    Compl z1(3, 4);   // 3 + 4i
    Compl z2(1, -2);  // 1 - 2i

    // Uncomment these as you implement each operator:
    // Compl sum = z1 + z2;
    // std::cout << "Sum: " << sum.getRe() << " + " << sum.getIm() << "i\n";
    // Expected: 4 + 2i

    // Compl prod = z1 * z2;
    // std::cout << "Product: " << prod.getRe() << " + " << prod.getIm() << "i\n";
    // Expected: (3*1 - 4*(-2)) + (3*(-2) + 4*1)i = 11 + (-2)i

    // std::cout << "Equal? " << (z1 == z2) << "\n";  // 0
    // std::cout << "Equal? " << (z1 == Compl(3, 4)) << "\n";  // 1

    // If you did the bonus:
    // std::cout << z1 << "\n";  // 3 + 4i
    // std::cout << z2 << "\n";  // 1 - 2i
}